# Challenge
In this lab, the objective is to identify the customers who were active in both May and June, and how did their activity differ between months. To achieve this, follow these steps:

1. Establish a connection between Python and the Sakila database.

2. Write a Python function called rentals_month that retrieves rental data for a given month and year (passed as parameters) from the Sakila database as a Pandas DataFrame. The function should take in three parameters:
    1. engine: an object representing the database connection engine to be used to establish a connection to the Sakila database.
    2. month: an integer representing the month for which rental data is to be retrieved.
    3. year: an integer representing the year for which rental data is to be retrieved.
   
   The function should execute a SQL query to retrieve the rental data for the specified month and year from the rental table in the Sakila database, and return it as a pandas DataFrame.

3. Develop a Python function called rental_count_month that takes the DataFrame provided by rentals_month as input along with the month and year and returns a new DataFrame containing the number of rentals made by each customer_id during the selected month and year.

4. The function should also include the month and year as parameters and use them to name the new column according to the month and year, for example, if the input month is 05 and the year is 2005, the column name should be "rentals_05_2005". Hint: Consider making use of pandas groupby()

4. Create a Python function called compare_rentals that takes two DataFrames as input containing the number of rentals made by each customer in different months and years. The function should return a combined DataFrame with a new 'difference' column, which is the difference between the number of rentals in the two months.

In [7]:
import pandas as pd
import numpy as np
import pymysql
from sqlalchemy import create_engine
from sqlalchemy import text
import getpass

password = getpass.getpass()

db = "sakila"
connection_string = 'mysql+pymysql://root:' + password + '@localhost/'+db
engine = create_engine(connection_string)

 ········


In [9]:
type(engine)

sqlalchemy.engine.base.Engine

In [65]:
def rentals_month(month, year, engine = engine):
    connection = engine.connect()
    query = text(f"""SELECT *
                    FROM rental
                    WHERE YEAR(rental_date) = {year}
                    		AND MONTH(rental_date) = {month}""")
    result = connection.execute(query)
    df = pd.DataFrame(result)
    return df

df = rentals_month(month = 5, year = 2005)
df

,rental_id,rental_date,inventory_id,customer_id,return_date,staff_id,last_update
0,1,2005-05-24 22:53:30,367,130,2005-05-26 22:04:30,1,2006-02-15 21:30:53
1,2,2005-05-24 22:54:33,1525,459,2005-05-28 19:40:33,1,2006-02-15 21:30:53
2,3,2005-05-24 23:03:39,1711,408,2005-06-01 22:12:39,1,2006-02-15 21:30:53
3,4,2005-05-24 23:04:41,2452,333,2005-06-03 01:43:41,2,2006-02-15 21:30:53
4,5,2005-05-24 23:05:21,2079,222,2005-06-02 04:33:21,1,2006-02-15 21:30:53
...,...,...,...,...,...,...,...
1151,1153,2005-05-31 21:36:44,2725,506,2005-06-10 01:26:44,2,2006-02-15 21:30:53
1152,1154,2005-05-31 21:42:09,2732,59,2005-06-08 16:40:09,1,2006-02-15 21:30:53
1153,1155,2005-05-31 22:17:11,2048,251,2005-06-04 20:27:11,2,2006-02-15 21:30:53
1154,1156,2005-05-31 22:37:34,460,106,2005-06-01 23:02:34,2,2006-02-15 21:30:53


In [66]:
def rental_count_month (data, month, year):
    df = pd.DataFrame(data["customer_id"].value_counts())
    column_name = f"rentals_{month}_{year}"
    df = df.rename({"count" : column_name}, axis = "columns")
    return df

df = rental_count_month (df, 5, 2005)
df

,rentals_5_2005
customer_id,
197,8
506,7
109,7
269,6
239,6
...,...
431,1
351,1
10,1


In [84]:
df1 = rentals_month(month = 5, year = 2005)
df2 = rentals_month(month = 6, year = 2005)

In [85]:
df1 = rental_count_month (df1, 5, 2005)
df2 = rental_count_month (df2, 6, 2005)

In [92]:
def compare_rentals (data1, data2):
    df = pd.merge(df1, df2, on='customer_id', how="outer")
    df.fillna(0, inplace = True)
    df["difference"] = df.iloc[:, 1] - df.iloc[:, 0]
    return df

df = compare_rentals(df1, df2)

In [93]:
df

,rentals_5_2005,rentals_6_2005,difference
customer_id,,,
1,2.0,7.0,5.0
2,1.0,1.0,0.0
3,2.0,4.0,2.0
4,0.0,6.0,6.0
5,3.0,5.0,2.0
...,...,...,...
595,1.0,2.0,1.0
596,6.0,2.0,-4.0
597,2.0,3.0,1.0
